# <center>大模型RAG进阶多格式文档解析

## 第一阶段：基础认知与准备

### 1. RAG基础知识回顾

首先回顾之前的RAG入门内容和基础架构：

<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015134735612.png" width="500">
</div>

**通用大模型的局限性：**

* 知识可能过时：大语言模型的训练数据都是存在时效性
* 会产生“幻觉”：生成的内容看似合理，但实际上与既定事实、真实数据或逻辑相悖
* 无法访问私有知识库数据：本身学习不到企业或者个人的私有知识库知识
* 回答缺乏具体出处：调研场景下，回答的内容给出具体的出处，一般是文章、论文等资料
* 最大对话上下文限制：大部分模型上下文限制还是128k左右

**RAG的核心要素：**

* 检索增强生成（**R**etrieval + **A**ugmented + **G**eneration）
* 为LLM提供了从某些数据源检索到的信息，并基于此修正生成的答案
* 让大模型学会“**查资料**”后再回答问题，而不是仅凭记忆回答
* 优势
    - 灵活性、适应性强
    - 提高大模型回答准确性
    - 成本相对低
    - 个性化程度高

### 2. 常见文档类型与解析需求

在真实企业环境中，会遇到各种数据源，特别是在金融领域的财报、报表等场景下，文档解析的需求尤为突出。

#### 常见的文档类型

<table style="width:100%; border-collapse: collapse; margin: 20px 0;">
    <style>
        table { border: 1px solid #ddd; }
        th { background-color: #4CAF50; color: white; padding: 12px; text-align: center; border: 1px solid #ddd; font-weight: bold; }
        td { padding: 10px; text-align: center; border: 1px solid #ddd; }
        tr:nth-child(even) { background-color: #f2f2f2; }
        tr:hover { background-color: #ddd; }
    </style>
    <thead>
        <tr>
            <th>扩展名称</th>
            <th>支持文件类型</th>
            <th>适用场景示例</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>csv</td>
            <td>.csv</td>
            <td>表格数据提取</td>
        </tr>
        <tr>
            <td>docx</td>
            <td>.doc, .docx</td>
            <td>Word 文档解析</td>
        </tr>
        <tr>
            <td>pdf</td>
            <td>.pdf</td>
            <td>PDF 文本/布局提取</td>
        </tr>
        <tr>
            <td>image</td>
            <td>.jpeg  .png  .tiff 等</td>
            <td>图片 OCR 文字识别</td>
        </tr>
        <tr>
            <td>pptx</td>
            <td>.ppt, .pptx</td>
            <td>幻灯片内容提取</td>
        </tr>
        <tr>
            <td>xlsx</td>
            <td>.xls, .xlsx</td>
            <td>Excel 表格解析</td>
        </tr>
    </tbody>
</table>

### 3. 文档解析挑战认知

#### 理解文档解析的难点

在考虑解析PDF文件时，我们需要根据当前的技术栈发展情况，并结合实际的业务诉求，综合考量这其中的技术难点，因为每一项技术难点所涉及的技术方案都会需要一个算法/或者技术手段去突破。

而开发者从解析的效果去考虑，可以从简单的做起，逐步突破难点，这对于开发人员自身的自信心提升也是一种正向的导向。在整个PDF解析过程中，以下几项是比较难处理的：

<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251022065557756.png" width="500">
</div>

* **布局解析困难**：PDF文件的布局可能会因为不同的作者、工具或用途而有所不同，通常具有多列文本，对于图像或表格来说，这些文本可能会突然中断，因此解析其布局是一个具有挑战性的任务。

* **格式错综复杂**：PDF文件中可能包含各种格式的内容，包括**文字**、**图像**、**表格**等，因此解析其内容需要考虑到这种多样性和复杂性。

* **复合表格**：纵向/横向合并的复杂表格，在PDF中进行抽象还原是最难处理的问题之一

* **文本、图片、表格顺序提取**：提取PDF文件中的文本、图片和表格，并确保它们的顺序正确性，是一个需要解决的重要问题。

* **文档结构还原**：还原PDF文件的文档结构，包括标题、目录等信息，是实现自动化文档处理和理解的关键步骤之一。

* **元数据提取**：在PDF中隐藏的元数据信息是RAG产品的关键数据，比如链接、目录、字体等等

* **扫描件**：PDF中如果是扫描件，就需要依靠OCR模型来进行有效的提取，这里面还会包含清晰度、模型的稳定性等等问题

* **Latex公式提取**：在一些特殊领域，PDF文本中包含了Latex等数学公式。通过完整的提取和转换是对RAG问答的有效补充

## 第二阶段：技术选型与工具对比

### 1. 主流工具对比与选择

**技术选型决策**
* **数字原生PDF**：优先选择PyMuPDF，利用其渲染效率优势处理纯文本/简单表格批量任务，只能处理纯文本

* **扫描PDF**：<font color='red'>必须启用OCR流程，可搭配Unstructured的"ocr_only"策略</font>

* **复杂学术文档**：推荐Marker（代码/公式支持）或MinerU（数学公式识别），但需容忍GPU加速需求

#### 工具性能对比表

<table style="width:100%; border-collapse: collapse; margin: 20px 0;">
    <style>
        table { border: 1px solid #ddd; }
        th { background-color: #4CAF50; color: white; padding: 12px; text-align: center; border: 1px solid #ddd; font-weight: bold; }
        td { padding: 10px; text-align: center; border: 1px solid #ddd; }
        tr:nth-child(even) { background-color: #f2f2f2; }
        tr:hover { background-color: #ddd; }
    </style>
    <thead>
        <tr>
            <th>工具</th>
            <th>核心优势</th>
            <th>适用场景</th>
            <th>性能代价</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>unstructured.io</td>
            <td>支持50+格式，生态完善</td>
            <td>多源数据ETL入口</td>
            <td>处理速度较慢</td>
        </tr>
        <tr>
            <td>PyMuPDF</td>
            <td>解析速度>200页/分钟</td>
            <td>纯文本/简单PDF批量处理</td>
            <td>无OCR能力</td>
        </tr>
        <tr>
            <td>Marker</td>
            <td>代码/公式支持优秀</td>
            <td>技术白皮书/学术文献</td>
            <td>需GPU加速</td>
        </tr>
        <tr>
            <td>MinerU</td>
            <td>数学公式识别精准</td>
            <td>科技/专利类文档</td>
            <td>高计算负载</td>
        </tr>
        <tr>
            <td>DoclingAI</td>
            <td>表格提取精度98%+</td>
            <td>金融财报/科研报告</td>
            <td>仅专注表格</td>
        </tr>
        <tr>
            <td>DeepDoc</td>
            <td>中文优化+端到端方案</td>
            <td>中文RAG系统建设</td>
            <td>需API调用</td>
        </tr>
    </tbody>
</table>

### 2. 文档解析技术差异

#### PDF解析技术核心差异

**PDF解析技术的核心差异体现在对文档结构的处理逻辑上：**
* **PyMuPDF**：渲染优先策略，高效文本提取，采用渲染优先策略，通过直接解析页面绘制指令实现高效文本提取；并非直接去“寻找”那些对人类而言可读的文字（**内容流**），而是像一名打印机，通过理解和执行模拟页面**渲染（绘制）过程**，执行底层绘制命令，来**重建页面的内容**，并从中精准定位和识别文本
    - 处理速度可达200页/分钟，尤其在规则表格识别中表现出坐标精度优势
    - 但缺乏OCR能力，无法处理扫描生成的图像型PDF，且对复杂公式和代码块的支持有限

* **OCR技术**：处理扫描件的核心能力
    - 针对无文本层的扫描件或多列布局文档，通过OCR提取文字

#### <font style="color:rgb(36, 41, 46);">OCR生态/大模型</font>

<font style="color:rgb(36, 41, 46);">OCR（</font><font style="color:rgb(15, 17, 21);">光学字符识别</font><font style="color:rgb(36, 41, 46);">）最终的目的是</font>**<font style="color:rgb(15, 17, 21);">将非结构化的图像信息，转化为结构化的、可计算和可理解的数据</font>**<font style="color:rgb(36, 41, 46);">，所以本质上是对图片内容的理解，可以考虑的开源组件如下：</font>

<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251022065557724.png" width="600">
</div>

#### Unstructured.io的集成优势

Unstructured.io作为集成框架，通过strategy参数实现后端自适应切换：
* **"fast"策略**：调用PyMuPDF等轻量引擎处理规则文档

* **"hi_res"策略**：激活YOLOX目标检测模型进行布局分析，配合detectron2实现表格与图像的精准提取

* **"ocr_only"策略**：使用OCR模型进行图文识别

* **"vlm"策略**：针对极端复杂场景调用GPT-4o等多模态模型，通过视觉语义理解突破传统解析局限

这种混合架构使其在金融财报（表格提取精度98%+）和科研报告等场景中表现突出。

#### **Unstructured的核心优势**

* 可以和Agent框架集成（LangChain、LlamaIndex）

* 也可以解析多种不同的文档形式（比较通用）

* 主流应用的比较多、适用性比较广

* **生态协同核心地位**：Unstructured库通过与LangChain主流框架的深度集成（如UnstructuredLoader组件），与LlamaIndex的深度集成（如UnstructuredReader组件）已成为检索增强生成（RAG）pipeline中的关键预处理节点。其能够将非结构化数据转化为向量数据库可索引的结构化格式，有效解决了RAG系统中"数据输入异构性"这一核心痛点，为下游的知识检索与生成任务提供了标准化数据基础

## 第三阶段：unstructured.io库入门与实践

### 1. 环境准备与安装

```bash
推荐使用Python 3.9及以上版本
```

#### （1）基本安装（纯文本处理）

**此安装方式适用于处理纯文本：**
```bash
pip install unstructured
uv add unstructured
```

#### （2）全量安装（多类型文档处理）

针对需要处理多类型文档（如PDF、Office格式、图片等）的场景，全量安装会包含docx、pptx、pdf、image等扩展依赖，适合企业级全场景文档处理需求：

**包含本地推理能力**（支持PDF/图片OCR等）
```bash
pip install "unstructured[local-inference]"
```

**支持所有文档类型**（不含本地推理，需依赖外部API）
```bash
pip install "unstructured[all-docs]"
uv add "unstructured[all-docs]"
```

#### （3）特定文档类型安装

如需进一步精简依赖，可按目标文件类型单独安装扩展模块，格式为`unstructured[<extra>]`，支持同时指定多个扩展，以逗号分隔：

**仅安装PDF和DOCX**处理能力
```bash
pip install "unstructured[pdf,docx]"
```

#### （4）Serverless API安装

Serverless API通过优化处理流程，将文档处理启动时间从30分钟缩短至3秒以内，并支持多区域横向扩展，有效提升了高并发场景下的吞吐量：

```bash
pip install unstructured-client
```

#### （5）Docker安装

```bash
docker pull downloads.unstructured.io/unstructured-io/unstructured:latest
docker run -dt --name unstructured downloads.unstructured.io/unstructured-io/unstructured:latest
docker exec -it unstructured bash
```

### 2. 核心系统依赖配置

#### （1）Tesseract OCR：图像文本识别

提供图像文本识别能力，是处理扫描版PDF和图片文件的核心组件。

* 安装指南：[https://tesseract-ocr.github.io/](https://tesseract-ocr.github.io/)
* 多语言支持：tesseract-lang扩展包

**windows安装：**

* 下载安装包：https://github.com/UB-Mannheim/tesseract/wiki
* 安装文档参考：https://developer.baidu.com/article/detail.html?id=3803413

**macOS安装：**
```bash
brew install tesseract
brew install tesseract-lang
```

**Linux安装：**
```bash
sudo apt-get install tesseract-ocr
sudo apt-get install tesseract-ocr-chi-sim  # 中文简体支持
```

**验证安装：**
```bash
tesseract --list-langs  # 查看已安装的语言包
```

In [ ]:
!tesseract -v

tesseract 5.2.0
 leptonica-1.82.0
  libgif 5.2.2 : libjpeg 9f : libpng 1.6.39 : libtiff 4.7.0 : zlib 1.2.13 : libwebp 1.3.2 : libopenjp2 2.5.2
 Found NEON
 Found libarchive 3.8.1 zlib/1.3.1 liblzma/5.6.4 bz2lib/1.0.8 liblz4/1.9.4 libzstd/1.5.7 libxml2/2.13.8 CommonCrypto/system libb2/bundled


#### （2）Poppler：PDF内容提取底层引擎

通过pdf2image库将PDF转换为图像格式，为后续OCR处理提供输入。

* 安装配置：[https://pdf2image.readthedocs.io/](https://pdf2image.readthedocs.io/)

**macOS安装：**
```bash
brew install poppler
```

**Linux安装：**
```bash
sudo apt-get install poppler-utils
```

**验证安装：**


In [ ]:
import os

# mac系统
# 设置 poppler 工具路径到环境变量
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ.get("PATH", "")

# windows系统
# 将此处路径替换为你自己的poppler\bin目录路径
#poppler_path = "C:\\Poppler\\bin"

# 将poppler路径临时添加到当前会话的环境变量中，os.pathsep 是自动添加路径分隔符（在Windows上是分号;）
#os.environ["PATH"] = poppler_path + os.pathsep + os.environ.get("PATH", "")

!pdfinfo -v


pdfinfo version 25.10.0
Copyright 2005-2025 The Poppler Developers - http://poppler.freedesktop.org
Copyright 1996-2011, 2022 Glyph & Cog, LLC


#### （3）Pandoc：富文本格式转换

处理EPUB、RTF等富文本格式的转换工具，**必须使用2.14.2及以上版本**以确保RTF文件解析兼容性。

#### （4）libmagic：跨平台文件类型检测

Linux和macOS系统需手动安装，Windows环境可忽略此依赖。

#### （5）常见依赖问题解决方案

**注意事项**：本地完整安装可能触发依赖链报错（如"Could not build wheels for pikepdf"），需预先安装qpdf、libheif和pillow等图像处理依赖。

### 3. unstructured核心功能理解

#### 功能分类与作用

**一般来说，这些功能分为几类：**

* **分区(Partitioning)**：将原始文档分解为标准的结构化元素

* **清理(Cleaning)**：从文档中删除不需要的文本，例如样板文件和句子片段

* **暂存(Staging)**：函数格式化下游任务的数据，例如ML推理和数据标记

* **分块(Chunking)**：功能将文档分割成更小的部分，以便在RAG应用程序和相似性搜索中使用

* **嵌入(Embedding)**：编码器类提供了一个接口，可以轻松地将预处理的文本转换为向量

In [ ]:
# UnstructuredIO核心组件
from unstructured.partition.auto import partition
from typing import List
from unstructured.documents.elements import Element

# 使用partition函数自动检测文件类型并解析,默认strategy策略是auto，还会有fast策略，速度比image-to-text models的快100倍
elements: List[Element] = partition(filename="RAG评估.md", strategy="auto")

# 元素的文本内容
print(elements[0].text)
print("===========================")

# 元素的类型
print(elements[0].category)
print("==================")

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")


2025-10-22 17:07:35,814 - WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


一.RAG效果评估的必要性
Title
{'category_depth': 0, 'languages': ['zho'], '_known_field_names': frozenset({'data_source', 'file_directory', 'orig_elements', 'page_number', 'is_continuation', 'subject', 'image_path', 'page_name', 'sent_to', 'category_depth', 'filename', 'emphasized_text_tags', 'image_url', 'header_footer_type', 'filetype', 'parent_id', 'bcc_recipient', 'link_start_indexes', 'emphasized_text_contents', 'sent_from', 'key_value_pairs', 'links', 'text_as_html', 'email_message_id', 'link_texts', 'image_mime_type', 'attached_to_filename', 'link_urls', 'last_modified', 'url', 'signature', 'cc_recipient', 'detection_class_prob', 'table_as_cells', 'languages', 'image_base64', 'detection_origin', 'coordinates'}), 'filename': 'RAG评估.md', 'filetype': 'text/markdown', 'last_modified': '2025-10-22T08:17:55'}


#### 基于元素的方法优势

为什么这种基于元素的方法如此重要？

* **结构为王**：通过将文档分解为这些语义元素，可以保留原始文档的大部分逻辑结构。您获得的不仅仅是原始文本；你得到的是带有上下文的文本
* **精细控制**：您可以迭代这些元素，按类型过滤（例如，"过滤所有Table元素"），或者根据它们的类别以不同的方式处理它们
* **丰富的元数据**：每个元素都包含有用的元数据：其文本内容、它来自的页码、通常是它在页面上的坐标（边界框）、原始文件名、HTML格式的元素以及检测到的语言。这允许精确的下游处理或链接回源

#### 元数据的应用价值

这些元数据让你能够：

* **精确定位**：知道文本在PDF中的确切位置
* **页面管理**：按页码组织和检索内容
* **多语言处理**：根据语言选择合适的处理策略
* **布局理解**：利用坐标信息进行版面分析

从本质上讲，unstructured不仅仅是"读取"PDF文档；它理解文档并进行解构它，这对于基于正则表达式来进行抓取的方式来说，这种解析方式无疑是一种对文档的更强大、更具语义性的理解。

### 4. Partition功能实践

**partition通用参数如下**：

* **encoding**：指定输入文本／文档读取时使用的字符编码。对于非 UTF-8 文档非常有用

* **include_page_breaks**：如果设置为 True，当文档支持 “分页” 时，输出中会包含 PageBreak 元素，以标识不同页的边界 ,markdown文档不支持分页

* **strategy**：指定解析策略，尤其对于 PDF/Image 文档，控制“快速 vs 高保真 vs OCR”方式

* **ocr_languages/languages**：当文档含有图像文字或扫描件时，可指定 OCR 语言包，如 ["eng","deu"]

* **skip_infer_table_types**：可指定跳过表格类型推断的文档类型，减少表格识别错误

* **fields_include**：控制输出 JSON 中包含哪些字段。可用于减小输出大小或过滤敏感字段["element_id","text","type","metadata"]

* **metadata_include / metadata_exclude**：用于控制在输出元素的 metadata 字段中，保留哪些键或者排除哪些键，默认全部输出

* **content_type**：在使用 URL 或文件流时，指定 MIME 类型提示，提高文件类型识别准确性

* **starting_page_number**：当处理文档是某个较大文档的一部分时，可以指定起始页号，用于 metadata

In [ ]:
from typing import List, Dict, Any, Optional, Sequence
from pathlib import Path

# 自定义解析函数，支持任意类型的文件格式
def parse_file_with_unstructured(file_path: str):
    """
    使用UnstructuredIO解析单个文件

    Args:
        file_path: 文件路径

    Returns:
        Dict: 包含解析结果和统计信息的字典
    """
    print(f"\n 解析文件: {file_path}")

    try:
        # 使用partition函数自动检测文件类型并解析,默认strategy策略是auto，还会有fast策略，速度比image-to-text models的快100倍
        elements: List[Element] = partition(filename=file_path, strategy="auto")

        # 分析解析结果
        analysis = {
            "file_path": file_path,
            "file_extension": Path(file_path).suffix.lower(),
            "total_elements": len(elements),
            "element_types": {},
            "elements": elements,
            "text_content": "",
            "statistics": {}
        }

        # 统计元素类型
        for element in elements:
            element_type = type(element).__name__
            analysis["element_types"][element_type] = analysis["element_types"].get(element_type, 0) + 1

        # 提取文本内容
        text_parts = []

        for element in elements:
            if hasattr(element, 'text') and element.text:
                text_parts.append(element.text)

        analysis["text_content"] = "\n\n".join(text_parts)

        # 计算统计信息
        analysis["statistics"]["total_characters"] = len(analysis["text_content"])

        print(f"   解析完成")
        print(f"   元素总数: {analysis['total_elements']}")
        print(f"   元素类型: {analysis['element_types']}")
        print(f"   总字符数: {analysis['statistics']['total_characters']}")
        print(f"   文本内容: {analysis['text_content'][:200]} ")

    except Exception as e:
        print(f"文件解析失败: {e}")
        return {}

#### （1）Markdown文档解析

In [ ]:
parse_file_with_unstructured("RAG评估.md")


libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.



 解析文件: RAG评估.md
   解析完成
   元素总数: 8
   元素类型: {'Title': 7, 'Table': 1}
   总字符数: 474
   文本内容: 一.RAG效果评估的必要性

二.RAG评估方法

1.人工评估

2.自动化评估

3.LangSmith

4.RAGAS

维度 LangSmith RAGAS 核心定位 大模型应用的 集成开发平台 (调试、测试、评估、监控) 专门的RAG评估框架 ，用于量化RAG管道在不同组件层面上的性能 核心功能 提供全链路功能：应用 调试、测试、评估、监控 专注于评估 ，提供针对RAG的专用评估指标  


In [1]:
from unstructured.partition.md import partition_md
from typing import List
from unstructured.documents.elements import Element

# 使用partition_md函数检测markdown文件类型解析,include_page_breaks若希望在 Markdown 中标识页面断点（少见场景）
elements: List[Element] = partition_md(filename="RAG评估.md", languages=["zho"],include_page_breaks=True)

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text)
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

ModuleNotFoundError: No module named 'unstructured'

#### （2）HTML文档解析

In [ ]:
parse_file_with_unstructured("html-tags-decode.html")


libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.



 解析文件: html-tags-decode.html
   解析完成
   元素总数: 4
   元素类型: {'Title': 1, 'NarrativeText': 2, 'Text': 1}
   总字符数: 232
   文本内容: 识别和解析HTML标签

HTML tags (filter) decode, You can increase safety by filtering the danger label.

注：虽然此功能能极大地扩展 Markdown 语法，但也面临着安全上的风险，所以默认是不开启的。

Update: 可以通过设置 `settings.htmlDecode = "style,script,if 



**支持 URL 输入、headers、ssl 验证选项**

* 除通用参数外：

    - url：直接给出网页 URL，无需先下载。

    - headers：HTTP 请求头（User-Agent 等）。

    - ssl_verify：是否验证 SSL 证书（False 可用于测试环境）。

    - file/filename/text：支持本地文件、文件流或网页文本输入。

    - content_type：可指定 text/html。

In [ ]:
from unstructured.partition.html import partition_html
from typing import List
from unstructured.documents.elements import Element

# 使用partition_html函数检测html网页类型解析
elements = partition_html(url="https://docs.unstructured.io/welcome",
                          headers={"User-Agent":"MyBot"},
                          ssl_verify=False,
                          include_page_breaks=False,
                          encoding="utf-8")
#elements: List[Element] = partition_html(url="https://docs.unstructured.io/welcome", languages=["zho"])

# 元素的元数据
print(elements[1].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[1].text)
print("===========================")

# 元素的类型
print(elements[1].category)
print("===========================")

{'link_texts': ['Unstructured home page'], 'link_urls': ['/'], 'languages': ['eng'], '_known_field_names': frozenset({'data_source', 'file_directory', 'orig_elements', 'page_number', 'is_continuation', 'subject', 'image_path', 'page_name', 'sent_to', 'category_depth', 'filename', 'emphasized_text_tags', 'image_url', 'header_footer_type', 'filetype', 'parent_id', 'bcc_recipient', 'link_start_indexes', 'emphasized_text_contents', 'sent_from', 'key_value_pairs', 'links', 'text_as_html', 'email_message_id', 'link_texts', 'image_mime_type', 'attached_to_filename', 'link_urls', 'last_modified', 'url', 'signature', 'cc_recipient', 'detection_class_prob', 'table_as_cells', 'languages', 'image_base64', 'detection_origin', 'coordinates'}), 'filetype': 'text/html', 'url': 'https://docs.unstructured.io/welcome'}
Unstructured home page
UncategorizedText


/opt/anaconda3/envs/rag_unstr/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'docs.unstructured.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


#### （3）EXCEL文档解析

In [ ]:
parse_file_with_unstructured("销售数据统计.xlsx")



 解析文件: 销售数据统计.xlsx
   解析完成
   元素总数: 1
   元素类型: {'Table': 1}
   总字符数: 1940
   文本内容: 日期 销售人员ID 销量 销售金额 10/20/2018 3 100 300 10/14/2018 4 100 100 12/20/2018 5 400 1200 10/23/2018 2 300 900 11/16/2018 3 100 100 10/30/2018 5 400 800 11/5/2018 5 100 200 12/28/2018 1 100 300 11/20/2018 1 1 


* 表格处理，支持多个 Sheet。

* 可调参数仍为通用那些（如 encoding、include_page_breaks）。

In [ ]:
from unstructured.partition.xlsx import partition_xlsx
from typing import List
from unstructured.documents.elements import Element

# 使用partition_xlsx函数检测excel文件类型并解析
elements: List[Element] = partition_xlsx(filename="销售数据统计.xlsx", languages=["zho"])

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text)
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

{'filename': '销售数据统计.xlsx', 'last_modified': '2025-10-20T14:22:04', 'page_name': 'Sheet1', 'page_number': 1, 'text_as_html': '<table><tr><td>日期</td><td>销售人员ID</td><td>销量</td><td>销售金额</td></tr><tr><td>10/20/2018</td><td>3</td><td>100</td><td>300</td></tr><tr><td>10/14/2018</td><td>4</td><td>100</td><td>100</td></tr><tr><td>12/20/2018</td><td>5</td><td>400</td><td>1200</td></tr><tr><td>10/23/2018</td><td>2</td><td>300</td><td>900</td></tr><tr><td>11/16/2018</td><td>3</td><td>100</td><td>100</td></tr><tr><td>10/30/2018</td><td>5</td><td>400</td><td>800</td></tr><tr><td>11/5/2018</td><td>5</td><td>100</td><td>200</td></tr><tr><td>12/28/2018</td><td>1</td><td>100</td><td>300</td></tr><tr><td>11/20/2018</td><td>1</td><td>100</td><td>200</td></tr><tr><td>10/16/2018</td><td>4</td><td>400</td><td>800</td></tr><tr><td>11/26/2018</td><td>2</td><td>200</td><td>400</td></tr><tr><td>10/25/2018</td><td>3</td><td>200</td><td>200</td></tr><tr><td>11/29/2018</td><td>1</td><td>400</td><td>800</td></tr><t

#### （4）CSV文档解析

In [ ]:
parse_file_with_unstructured("训练数据.csv")


libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.



 解析文件: 训练数据.csv
   解析完成
   元素总数: 1
   元素类型: {'Table': 1}
   总字符数: 186191
   文本内容: months_as_customer age policy_number policy_bind_date policy_state policy_csl policy_deductable policy_annual_premium umbrella_limit insured_zip insured_sex insured_education_level insured_occupation  


* 相对简单，主要表格抽取。

* 可调参数少，通常只使用通用参数如 encoding、include_page_breaks。

* 输出为一个 Table 元素，其 metadata.text_as_html 包含 HTML 表格。

In [ ]:
from unstructured.partition.csv import partition_csv
from typing import List
from unstructured.documents.elements import Element

# 使用partition_csv函数检测csv文件类型并解析
elements = partition_csv(filename="训练数据.csv", encoding="utf-8")

# 元素的元数据
#print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text[:400])
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

months_as_customer age policy_number policy_bind_date policy_state policy_csl policy_deductable policy_annual_premium umbrella_limit insured_zip insured_sex insured_education_level insured_occupation insured_hobbies insured_relationship capital-gains capital-loss incident_date incident_type collision_type incident_severity authorities_contacted incident_state incident_city incident_location incide
Table


#### （5）Word文档解析

In [ ]:
parse_file_with_unstructured("数组.docx")


 解析文件: 数组.docx
   解析完成
   元素总数: 153
   元素类型: {'Title': 21, 'NarrativeText': 13, 'Text': 44, 'ListItem': 75}
   总字符数: 7656
   文本内容: 1. 数组简介 

1.1 概述

我们之前学习的变量或者是常量, 只能用来存储一个数据, 例如: 存储一个整数, 小数或者字符串等. 如果需要同时存储多个同类型的数据, 用变量或者常量来实现的话, 非常的繁琐. 针对于这种情况, 我们就可以通过数组来实现了.

例如: 假设某公司有50名员工, 现在需要统计该公司员工的工资情况, 例如计算平均工资、获取最高工资等。针对于这个需求，如果用前面所学的 


* 支持 .docx（样式信息）和 .doc（需 LibreOffice 转换）。

* 参数：同通用参数。

In [ ]:
from unstructured.partition.docx import partition_docx
from unstructured.partition.doc import partition_doc
from typing import List
from unstructured.documents.elements import Element

# 使用partition_docx函数检测word文件类型并解析，include_page_breaks当文档支持 “分页” 时，以标识不同页的边界
elements = partition_docx(filename="数组.docx", encoding="utf-8", include_page_breaks=True)

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text[:400])
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

{'category_depth': 2, 'filename': '数组.docx', 'last_modified': '2025-10-20T14:20:10', 'languages': ['zho'], '_known_field_names': frozenset({'data_source', 'file_directory', 'orig_elements', 'page_number', 'is_continuation', 'subject', 'image_path', 'page_name', 'sent_to', 'category_depth', 'filename', 'emphasized_text_tags', 'image_url', 'header_footer_type', 'filetype', 'parent_id', 'bcc_recipient', 'link_start_indexes', 'emphasized_text_contents', 'sent_from', 'key_value_pairs', 'links', 'text_as_html', 'email_message_id', 'link_texts', 'image_mime_type', 'attached_to_filename', 'link_urls', 'last_modified', 'url', 'signature', 'cc_recipient', 'detection_class_prob', 'table_as_cells', 'languages', 'image_base64', 'detection_origin', 'coordinates'}), 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document'}
1. 数组简介 
Title


#### （6）Image图片解析

**模型下载说明**

* 在对Image图片进行解析时，默认下载的Yolox模型来进行识别。默认是从huggingface上下载模型，所以需要科学上网。

* 安装完poppler-utils、tesseract-ocr后，成功下载yolox模型即可正常使用。

<dev align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251022115221931.png" width="700">
</div>

<dev align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251022115221934.png" width="700">
</div>

In [ ]:
parse_file_with_unstructured("PDF解析截图.png")


 解析文件: PDF解析截图.png
   解析完成
   元素总数: 23
   元素类型: {'Header': 2, 'Title': 6, 'NarrativeText': 8, 'Image': 2, 'Text': 5}
   总字符数: 1746
   文本内容: = O.LangChaintX REAM. pdf

3 ngChain RRA I] SAgentFASLAK-Part

—, LangChain.aiLRtAa7713

AHIATHR, BAAR EHTS MAgentH RL LangChain,

1. GPT-3RiC FST ARBFALA

Langchain ALAR Z AB 202247 ARBRE KARAS —TAIE 


* 类似于 PDF 的策略调用，但直接针对单张或多张图片。

* 特定参数：

    - strategy：可选 "auto", "hi_res", "ocr_only"。

    - languages / ocr_languages：OCR 多语言支持。

    - include_page_breaks：如图片列表可视为 “页”。

In [ ]:
from unstructured.partition.image import partition_image
from typing import List
from unstructured.documents.elements import Element

# 使用partition_image函数检测png类型并解析，strategy="ocr_only"使用ocr来进行图片内容文字识别
elements = partition_image(filename="PDF解析截图.png",
                           strategy="ocr_only",
                           languages=["eng","chi_sim"],
                           include_page_breaks=False)

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text[:400])
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

{'coordinates': CoordinatesMetadata(points=((np.float64(50.0), np.float64(39.0)), (np.float64(50.0), np.float64(77.0)), (np.float64(2424.0), np.float64(77.0)), (np.float64(2424.0), np.float64(39.0))), system=<unstructured.documents.coordinates.PixelSpace object at 0x34d987af0>), 'filetype': 'PNG', 'languages': ['eng', 'zho'], 'last_modified': '2025-10-20T10:15:27', 'page_number': 1, '_known_field_names': frozenset({'data_source', 'file_directory', 'orig_elements', 'page_number', 'is_continuation', 'subject', 'image_path', 'page_name', 'sent_to', 'category_depth', 'filename', 'emphasized_text_tags', 'image_url', 'header_footer_type', 'filetype', 'parent_id', 'bcc_recipient', 'link_start_indexes', 'emphasized_text_contents', 'sent_from', 'key_value_pairs', 'links', 'text_as_html', 'email_message_id', 'link_texts', 'image_mime_type', 'attached_to_filename', 'link_urls', 'last_modified', 'url', 'signature', 'cc_recipient', 'detection_class_prob', 'table_as_cells', 'languages', 'image_base6

In [ ]:
elements = partition_image(filename="数学公式.png",
                           strategy="ocr_only",
                           languages=["eng","chi_sim"],
                           include_page_breaks=False)

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text[:400])
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

{'coordinates': CoordinatesMetadata(points=((np.float64(23.0), np.float64(66.0)), (np.float64(23.0), np.float64(97.0)), (np.float64(283.0), np.float64(97.0)), (np.float64(283.0), np.float64(66.0))), system=<unstructured.documents.coordinates.PixelSpace object at 0x3858e8850>), 'filetype': 'PNG', 'languages': ['eng', 'chi_sim'], 'last_modified': '2025-10-20T20:50:51', 'page_number': 1, '_known_field_names': frozenset({'data_source', 'file_directory', 'orig_elements', 'page_number', 'is_continuation', 'subject', 'image_path', 'page_name', 'sent_to', 'category_depth', 'filename', 'emphasized_text_tags', 'image_url', 'header_footer_type', 'filetype', 'parent_id', 'bcc_recipient', 'link_start_indexes', 'emphasized_text_contents', 'sent_from', 'key_value_pairs', 'links', 'text_as_html', 'email_message_id', 'link_texts', 'image_mime_type', 'attached_to_filename', 'link_urls', 'last_modified', 'url', 'signature', 'cc_recipient', 'detection_class_prob', 'table_as_cells', 'languages', 'image_bas

**需要注意**：

* OCR的识别效果很大程度上取决于原始图像的质量。如果识别结果不理想，可以尝试对图像进行预处理，例如**二值化、降噪、倾斜校正**等

#### （7）PDF文件解析

In [ ]:
parse_file_with_unstructured("甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf")


 解析文件: 甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf
   解析完成
   元素总数: 50
   元素类型: {'Title': 28, 'NarrativeText': 2, 'Header': 3, 'Text': 13, 'ListItem': 4}
   总字符数: 4273
   文本内容: 证 券 研 究 报 告

行 业 研 究

行 业 点 评

海外科技巨头持续发力 AI，龙头公司中报业绩亮眼

——AI 行业点评报告

◼ 核心观点 海外 AI 视角：（1）英伟达推出 B200A，2025 年 Blackwell GPU 有望 上量。《科创板日报》8 月 7 日讯，据 TrendForce 集邦咨询，英伟达仍 计划在 2024 年下半年推出 B100 及 B200，供应 CS 


* <font color='red'>关键额外参数：</font>

    - strategy：PDF解析策略的选择直接影响提取效果，可选 "auto"（默认）、"hi_res"、"fast"、"ocr_only"。控制解析方式。
        - **auto**（默认）：适用于无图像嵌入文本的标准PDF，解析速度快

        - **hi_res**：使用布局检测模型（例如 会调用布局检测模型 Detectron2/YOLOX/自研Chipper等）提取结构信息。

        - **fast**：快速解析，以可提取文本为主。

        - **ocr_only**：针对扫描件／图像版 PDF，只做 OCR 提取。

        - **vlm**：VLM模型，如OpenAI/Anthropic等提供的视觉语言模型

    - extract_images_in_pdf（布尔）：当 strategy 为 hi_res 时，可控制是否提取嵌入图像块。

    - extract_image_block_types（列表）：指定提取哪些类型（如 ["Image","Table"]）的图像块。

    - extract_image_block_to_payload（布尔）：是否把提取块转换为 payload（例如 base64）输出，这个是为了多模态准备的，有些内容就需要保存图片，不需要ocr。

    - extract_image_block_output_dir（字符串）：如果不转换为 payload，可将提取图像块保存到指定目录。

    - max_partition：当使用 ocr_only 策略时，限制单个元素（文本块）最大字符长度。默认为 1500。

    - languages 或 ocr_languages：对 OCR 使用的语言包列表。

    - skip_infer_table_types：可以跳过表格类型推断以提高速度。

    - split_pdf_page：True，大文件分块处理，可实现大文件分块处理，提升解析效率

    - infer_table_structure：True，表格结构推断，是否尝试推断表格结构

表格提取功能已集成至unstructured库核心模块，无需再向unstructured-inference传递extract_tables参数。通过elements对象的category属性可精准筛选表格元素。

In [ ]:
from unstructured.partition.pdf import partition_pdf
from typing import List
from unstructured.documents.elements import Element

# 使用partition_pdf函数检测pdf类型并解析
elements = partition_pdf(filename="甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf",
                         strategy="hi_res", # 使用hi_res模式进行高精度解析
                         extract_images_in_pdf=True, # 提取pdf中的图片
                         extract_image_block_types=["Table","Image"], # 提取表格和图片
                         extract_image_block_output_dir="./images", # 保存图片到images目录
                         languages=["eng","zho"],
                         split_pdf_page=True, # 大文件分块处理，优化性能
                         infer_table_structure=True, # 是否尝试推断表格结构，会下载一个视觉目标检测的 Transformer 模型
                         include_page_breaks=True) # 是否包含页码信息


# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text[:400])
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

{'detection_class_prob': 0.5647643208503723, 'coordinates': CoordinatesMetadata(points=((np.float64(13.965239524841309), np.float64(-6.397173881530762)), (np.float64(13.965239524841309), np.float64(858.3699951171875)), (np.float64(63.11024475097656), np.float64(858.3699951171875)), (np.float64(63.11024475097656), np.float64(-6.397173881530762))), system=<unstructured.documents.coordinates.PixelSpace object at 0x34b5fe260>), 'links': [], 'last_modified': '2025-10-14T16:26:25', '_known_field_names': frozenset({'data_source', 'file_directory', 'orig_elements', 'page_number', 'is_continuation', 'subject', 'image_path', 'page_name', 'sent_to', 'category_depth', 'filename', 'emphasized_text_tags', 'image_url', 'header_footer_type', 'filetype', 'parent_id', 'bcc_recipient', 'link_start_indexes', 'emphasized_text_contents', 'sent_from', 'key_value_pairs', 'links', 'text_as_html', 'email_message_id', 'link_texts', 'image_mime_type', 'attached_to_filename', 'link_urls', 'last_modified', 'url', '

<font color='red'>在加上infer_table_structure=True(是否尝试推断表格结构)参数以后会直接下载一个视觉目标检测的 Transformer 模型，用来在图上画线，区分别表格中的每个单元格 ，反正只要有表格，OCR还是少不了的</font>

开启infer_table_structure=True这个参数，就意味着你允许系统去下载并运行一套**“视觉检测（Transformer）+ 文字识别（OCR）”**的组合拳。

这也是为什么在处理单纯的电子版表格时，如果你觉得这套流程太重（下载慢、吃显存），算法工程师通常会考虑用 pdfplumber 这种纯规则解析方案来替代 unstructured，因为它不需要下载任何 Transformer 模型。

<dev align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251022115221928.png" width="500">
</div>

#### （8）Element对象核心字段


返回的Element对象包含以下核心字段：
* **text**：元素的文本内容（表格会以Markdown表格格式呈现）
* **category**：元素类型，如Title、NarrativeText、ListItem、Table等
* **metadata**：元素的元数据

#### （9）category元素类型详解

* **Title（标题）**：文档的标题和副标题
* **NarrativeText（叙事文本）**：纯文本的段落
* **Table（表格）**：表格数据
* **Text（段落）**：文本段落
* **Image（图像）**：所有图片
* **Formula（数学公式）**：文本 y = Wx + b
* **Header/Footer（页眉/页脚）**：可以将它们与主要内容区分开来

#### （10）metadata元数据详解

* **页码（page_number）**：该文本块所在的页码（从1开始）
* **坐标信息（coordinates）**：
* **points**：文本块在页面上的边界框坐标（4个角的坐标点）
    - 格式：`[左上, 左下, 右下, 右上]`
    - 单位：像素点
* **system**：坐标系统类型（PixelSpace = 像素坐标系）
* **layout_width/height**：页面的总宽度和高度（像素）
* **语言（languages）**：检测到的文档语言（如`["zho"]` = 中文，ISO 639-3语言代码）
* **文件基本信息**：
* **filename**：原始文件名
* **last_modified**：文件最后修改时间（ISO 8601格式）
* **filetype**：文件MIME类型（如`application/pdf`）

#### （11）下游RAG应用优化

**在检索增强生成（RAG）系统中：**
* 通过**category过滤非文本元素**（如图像）
* 或优先使用**标题元素构建文档层次结构**，提升检索准确性

#### （12）额外配置：指定OCR Agent

如果你想切换OCR引擎（例如用Paddle OCR做中文识别更好），可以在系统环境变量中设置`OCR_AGENT`：
 * 官方文档：https://docs.unstructured.io/open-source/how-to/set-ocr-agent?utm_source=chatgpt.com


**使用Tesseract（默认)**
* export OCR_AGENT="unstructured.partition.utils.ocr_models.tesseract_ocr.OCRAgentTesseract"

**或使用Paddle OCR（若已安装）**
* export OCR_AGENT="unstructured.partition.utils.ocr_models.paddle_ocr.OCRAgentPaddle"


并确保你安装了对应依赖（Tesseract二进制+语言包，或Paddle、Google SDK）。这能显著影响中文识别质量。

### 6. 常见问题与解决方案

**partition_pdf导入错误处理**
* **问题**：`partition_pdf`导入时报错或找不到模块（版本差异）
* **解决**：检查`unstructured`版本与文档示例是否匹配；有时接口结构在minor/patch版本中变更（例如partition_pdf的import路径）。必要时查看GitHub issues

**hi_res本地安装复杂问题**
* **问题**：`hi_res`本地安装复杂（detectron2在Windows安装困难）
* **解决**：使用`unstructured-api`的远程推理端点或者在Linux/GPU容器中部署`unstructured-inference`

**表格转换结果错位修正**
* **问题**：表格转换结果错位/合并不正确
* **解决**：尝试在`hi_res`下调整表格推断参数，或在解析后用pandas的post-processing（合并列/填充空值）修正；对复杂表格考虑手工清洗或专用表格抽取工具（Camelot、Tabula）做对比

**中文识别问题**
* **问题**：识别不到中文/乱码
* **解决**：确认Tesseract已安装相应语言包（`tesseract --list-langs`）；使用`languages=["chi_sim","eng"]`并重启进程

**混合中英文本方向/版式混乱**
* **解决**：优先使用`strategy="hi_res"`做布局检测，再对图片区域分别OCR；对低质量扫描先做图像预处理（deskew、denoise、binarize）

**性能优化策略（经验型）：**

* 在处理大量电子PDF（有文本层）时：`fast` + PyMuPDF/pdftotext管线通常最快且足够准确
* 在需要高质量表格边界或复杂布局时：`hi_res`能显著提高表格/标题/图片检测但开销大（时间/依赖）。若对吞吐量有严格要求，考虑把`hi_res`推理外包到推理服务并对任务做批量调度（异步/队列）

## 第四阶段：LlamaIndex框架介绍

### 1. 大模型开发框架（SDK）是什么？

_SDK：Software Development Kit，它是一组软件工具和资源的集合，旨在帮助开发者创建、测试、部署和维护应用程序或软件。_
所有开发框架（SDK）的核心价值，都是降低开发、维护成本。
大语言模型开发框架的价值，是让开发者可以更方便地开发基于大语言模型的应用。

**主要提供两类帮助：**

* 第三方能力抽象。比如LLM、向量数据库、搜索接口等

* 常用工具、方案封装

* 底层实现封装。比如流式接口、超时重连、异步与并行等

**好的开发框架，需要具备以下特点：**

* 可靠性、鲁棒性高

* 可维护性高

* 可扩展性高

* 学习成本低

**举些通俗的例子：**
* 与外部功能解依赖
* 比如可以随意更换LLM而不用大量重构代码
* 更换三方工具也同理
* 经常变的部分要在外部维护而不是放在代码里
* 比如Prompt模板
* 各种环境下都适用
* 比如线程安全
* 方便调试和测试
* 至少要能感觉到用了比不用方便吧
* 合法的输入不会引发框架内部的报错

**选对了框架，事半功倍；反之，事倍功半。**

**参考资源：**

* 什么是SDK? [https://aws.amazon.com/cn/what-is/sdk/](https://aws.amazon.com/cn/what-is/sdk/)
* SDK和API的区别是什么? [https://aws.amazon.com/cn/compare/the-difference-between-sdk-and-api/](https://aws.amazon.com/cn/compare/the-difference-between-sdk-and-api/)

### 2. LlamaIndex介绍


LlamaIndex 是一个为开发「上下文增强」的大语言模型应用的框架（也就是 SDK）。上下文增强，泛指任何在私有或特定领域数据基础上应用大语言模型的情况。例如：

<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251022065557750.png" width="600">
</div>

LlamaIndex 有 Python 和 Typescript 两个版本，Python 版的文档相对更完善。


* Python 文档地址：https://docs.llamaindex.ai/en/stable/

* Python API 接口文档：https://docs.llamaindex.ai/en/stable/api_reference/

* TS 文档地址：https://ts.llamaindex.ai/

* TS API 接口文档：https://ts.llamaindex.ai/api/

* LlamaIndex 是一个开源框架，Github 链接：https://github.com/run-llama

### 3.LlamaIndex 的核心模块

<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251022065557754.png" width="900">
</div>

### 4. 框架对比：LangChain vs LlamaIndex

<table style="width:100%; border-collapse: collapse; margin: 20px 0;">
    <style>
        table { border: 1px solid #ddd; }
        th { background-color: #4CAF50; color: white; padding: 12px; text-align: center; border: 1px solid #ddd; font-weight: bold; }
        td { padding: 10px; text-align: center; border: 1px solid #ddd; }
        tr:nth-child(even) { background-color: #f2f2f2; }
        tr:hover { background-color: #ddd; }
    </style>
    <thead>
        <tr>
            <th>对比维度</th>
            <th>LangChain</th>
            <th>LlamaIndex（原 GPT Index）</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>定位与设计</td>
            <td>通用型 LLM 应用框架（可构建 Agent、Tool、RAG 等各种系统）</td>
            <td>专注文档理解与 RAG 的索引、检索、问答框架</td>
        </tr>
        <tr>
            <td>核心理念</td>
            <td>“链式调用（Chains）” 和 “工具组合（Tools）”，强调可编排性</td>
            <td>“数据接口（Data Index + Query Engine）”，强调数据到知识的映射</td>
        </tr>
        <tr>
            <td>文档加载能力</td>
            <td>`DocumentLoader`<br/> 支持多格式文档（PDF、HTML、TXT）</td>
            <td>深度集成解析器（如 LlamaParse、Unstructured、PandasReader）</td>
        </tr>
        <tr>
            <td>数据解析深度</td>
            <td>主要提取纯文本，结构化需手动实现</td>
            <td>提供高层结构抽象（Document → Node → Index），保留层级关系</td>
        </tr>
        <tr>
            <td>索引结构</td>
            <td>VectorStore（Chroma、FAISS、Milvus等）为核心，开发者需手动管理</td>
            <td>提供多种索引：`VectorStoreIndex`<br/>, `SummaryIndex`<br/>, `KnowledgeGraphIndex`</td>
        </tr>
        <tr>
            <td>上下文压缩 / rerank</td>
            <td>需额外配置 Reranker 或 ContextCompressor</td>
            <td>原生支持 Context Compression / Node PostProcessor</td>
        </tr>
        <tr>
            <td>多文档检索</td>
            <td>可通过 `MultiQueryRetriever`<br/>、`ParentDocumentRetriever`<br/> 实现</td>
            <td>内置 `ComposableGraph`<br/> 实现多索引融合检索</td>
        </tr>
        <tr>
            <td>Agent 能力</td>
            <td>强（LangChain 是 Agent 生态核心框架）</td>
            <td>弱（更偏向数据检索与知识问答）</td>
        </tr>
        <tr>
            <td>生态与扩展性</td>
            <td>最大的 LLM 生态，插件、工具链最丰富</td>
            <td>与数据密集型 RAG 项目结合最紧（适合文档知识库类应用）</td>
        </tr>
        <tr>
            <td>适用场景</td>
            <td>多步骤推理、Agent系统、工具调用、企业助手</td>
            <td>文档问答、知识检索、企业知识库、研究型报告分析</td>
        </tr>
        <tr>
            <td>示例语法简洁度</td>
            <td>代码偏工程风格，组件组装较多</td>
            <td>封装层高，一行代码即可构建 query engine</td>
        </tr>
        <tr>
            <td>性能优化方向</td>
            <td>优化在链路编排与检索召回效率</td>
            <td>优化在文档解析、chunk 切分与上下文压缩</td>
        </tr>
        <tr>
            <td>代表项目</td>
            <td>Chatbot、智能助理、Agent系统、数据问答</td>
            <td>企业知识库问答、PDF报告分析、学术RAG系统</td>
        </tr>
    </tbody>
</table>

* **LangChain 是“逻辑大脑”**：强在工作流编排、Agent化、工具集成。

* **LlamaIndex 是“知识记忆”**：强在文档结构化、RAG索引、检索优化。

    - 两者并不是竞争关系，而是 **RAG 系统的天然组合**

## 第五阶段：LlamaIndex集成与进阶

### 1. LlamaIndex环境准备

**核心库安装**

**LlamaIndex版本0.14.x**

```bash
pip install llama-index-core llama-index
```

In [1]:
!pip install llama-index-core llama-index

**解析库安装**
```bash
pip install llama-parse unstructured nest-asyncio python-multipart llama-index-readers-file
pip install pytest
pip install "unstructured[md]"
```

In [1]:
!pip install llama-parse unstructured nest-asyncio python-multipart llama-index-readers-file
!pip install pytest
!pip install "unstructured[md]"

### **2. LlamaIndex常用组件**

**常用组件：**

* `SimpleDirectoryReader`
* `LlamaParse`（针对复杂 PDF）
* `UnstructuredReader`（多格式文档）
* `PandasReader`（表格类文件）
* 官方文档申请api_key：[https://llamaindex.org.cn/blog/pdf-parsing-llamaparse](https://llamaindex.org.cn/blog/pdf-parsing-llamaparse)

**示例：**

In [2]:
from llama_index.core import SimpleDirectoryReader
from llama_parse import LlamaParse

# 如果文档结构复杂，优先使用 LlamaParse
# parser = LlamaParse(api_key="YOUR_LLAMA_CLOUD_API_KEY")
# documents = parser.load_data("sample.pdf")

# 或者使用简单读取器
documents = SimpleDirectoryReader(input_files=["RAG评估.md"]).load_data()

print(documents[0].metadata)
print("===========================")
print(documents[0].text)
print("===========================")

/tmp/ipykernel_30852/3545504123.py:2: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


ValueError: File RAG评估.md does not exist.

### 3. LlamaIndex集成unstructured

#### LlamaIndex与Unstructured的关系

* LlamaIndex本身并不专注于文件解析（Parsing），而专注于：

    - "结构化地管理与大模型交互的外部知识（即索引、检索、问答）。"

    - 而Unstructured.io是一个独立的"文档解析引擎"，核心职责是：

    - "将各种复杂格式（PDF、DOCX、HTML、Excel、图片等）解析成统一的文本元素（elements）。"

因此：
* **LlamaIndex是知识管理层（Knowledge Layer）**

* **Unstructured是文档提取层（Extraction Layer）**

#### 两种集成方式对比

<table style="width:100%; border-collapse: collapse; margin: 20px 0;">
    <style>
        table { border: 1px solid #ddd; }
        th { background-color: #4CAF50; color: white; padding: 12px; text-align: center; border: 1px solid #ddd; font-weight: bold; }
        td { padding: 10px; text-align: center; border: 1px solid #ddd; }
        tr:nth-child(even) { background-color: #f2f2f2; }
        tr:hover { background-color: #ddd; }
    </style>
    <thead>
        <tr>
            <th>对比点</th>
            <th>使用`unstructured.partition`直接解析</th>
            <th>使用`LlamaIndex UnstructuredReader`</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>**底层调用**</td>
            <td>直接调用`unstructured`官方API（`partition()`）</td>
            <td>内部封装了`partition()`，简化调用</td>
        </tr>
        <tr>
            <td>**灵活度**</td>
            <td>可访问所有底层参数（如`strategy`、`hi_res_model`、`ocr_languages`）</td>
            <td>封装后部分参数隐藏，仅暴露常用接口</td>
        </tr>
        <tr>
            <td>**可控性**</td>
            <td>可自定义处理流程（过滤、正则、chunk策略）</td>
            <td>自动化程度高，但定制难度大</td>
        </tr>
        <tr>
            <td>**集成便捷性**</td>
            <td>需手动将结果转为`Document`</td>
            <td>自动输出为`Document`列表</td>
        </tr>
        <tr>
            <td>**依赖管理**</td>
            <td>由开发者决定何时安装哪些后端（pdfminer, tesseract）</td>
            <td>自动导入必要模块，错误提示更友好</td>
        </tr>
        <tr>
            <td>**适用场景**</td>
            <td>高级研发/多源异构文档处理</td>
            <td>快速原型/小规模项目</td>
        </tr>
    </tbody>
</table>


#### 方式一：直接使用UnstructuredReader

**推荐场景：快速测试/教学/单格式文件读取**

**优点：**
* 写法极简
* 自动生成Document对象
* 无需显式调用`partition`

**缺点：**
* 无法<font color='red'>细调</font>OCR(因为参数都被隐藏了)、chunk_size、文本清洗,
* 对图片、HTML、公式等复杂结构支持有限
* Document对象只保留了文本和元数据，没有数据类型


* 直接把所有的elements列表 合并成一个大Document对象

In [ ]:
from llama_index.readers.file.unstructured import UnstructuredReader
from pathlib import Path

reader = UnstructuredReader()
# 如果要控制是否启用 hi_res ，load_data中可以传参
documents = reader.load_data(file=Path("甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf"))

print("打印列表长度：" + str(len(documents)))
print("==================================")
print("打印解析的文本内容：" + documents[0].text[:100])
print("==================================")
print("打印元数据信息：" + str(documents[0].metadata))

2025-10-22 13:01:56,059 - WARNING - 'doc_id' is deprecated and 'id_' will be used instead


打印列表长度：1
打印解析的文本内容：证 券 研 究 报 告

行 业 研 究

行 业 点 评

海外科技巨头持续发力 AI，龙头公司中报业绩亮眼

——AI 行业点评报告

◼ 核心观点 海外 AI 视角：（1）英伟达推出 B200A
打印元数据信息：{'coordinates': '{"points": [[6.84, 80.35964000000001], [6.84, 163.39963999999975], [20.88, 163.39963999999975], [20.88, 80.35964000000001]], "system": "PixelSpace", "layout_width": 595.32, "layout_height": 841.92}', 'filename': '甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf', 'last_modified': '2025-10-14T16:26:25', 'page_number': 1, 'languages': '["zho"]', 'filetype': 'application/pdf'}


#### 方式二：独立使用unstructured.partition + 自定义逻辑

**推荐场景：生产级RAG应用/多格式数据管线/高可控性需求**

**优点：**
* 可自由控制解析策略（OCR、chunk、去噪、正则）
* 可在加载前后插入清洗逻辑（例如表格转结构化文本）
* 易于扩展（批量处理PDF/并行任务/自定义metadata）

In [ ]:
from unstructured.partition.auto import partition
# 使用LlamaIndex的Document对象，将解析后的元素转换为Document对象
from llama_index.core import Document

# 使用partition函数自动检测文件类型并解析
elements = partition(
    filename="甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf",
    strategy="hi_res",
    split_pdf_page=True,
    infer_table_structure=True,
    languages=["eng","chi_sim"])

# 将解析后的元素转换为Document对象
docs = [
    Document(text=e.text,
             metadata={"source":"甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf",
                       "type": e.category})
    for e in elements]


2025-10-22 13:44:56,364 - INFO - Reading PDF for file: 甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf ...


In [ ]:
docs

[Document(id_='3cc58f12-76fd-4db6-8ceb-a0a52d1b9065', embedding=None, metadata={'source': '甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf', 'type': 'Header'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='证 券 研 究 报 告 行 业 研 究', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}'),
 Document(id_='9a249685-3791-4ab2-a347-910ba502219a', embedding=None, metadata={'source': '甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf', 'type': 'Header'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='中小市值', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_te

#### 方式三：最佳混合方案（推荐实践）

结合两者优势：

In [ ]:
from llama_index.readers.file.unstructured import UnstructuredReader
from unstructured.partition.auto import partition
from llama_index.core import Document
from pathlib import Path

def smart_load(file_path):
    """
    智能文档加载器：根据文件类型选择最佳解析策略

    Args:
        file_path: 文件路径

    Returns:
        解析后的Document对象列表
    """
    file_path = Path(file_path)
    file_ext = file_path.suffix.lower()

    # 定义复杂文件类型（需要高精度解析）
    complex_types = {
        '.pdf',     # PDF文档（可能包含表格、图像、复杂布局）
        '.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff',  # 图片文件（需要OCR）
        '.docx', '.doc',  # Word文档（可能包含复杂格式）
        '.pptx', '.ppt',  # PowerPoint（复杂布局）
        '.xlsx', '.xls'   # Excel（表格结构）
    }

    # 简单文件类型（可以用Reader直接处理）
    simple_types = {
        '.txt', '.md', '.csv', '.html', '.xml', '.json'
    }

    if file_ext in complex_types:
        # 复杂文件使用底层解析，获得更好的结构识别
        print(f"检测到复杂文件类型 {file_ext}，使用partition高精度解析")
        try:
            elements = partition(
                filename=str(file_path),
                # 使用hi_res模式进行高精度解析
                strategy="hi_res",
                # 支持中文、英文
                languages=["eng", "chi_sim"],
                # 推断表格结构
                infer_table_structure=True
            )
            # 将解析元素转换为Document对象
            return [Document(text=e.text, metadata={
                "source": str(file_path),
                "element_type": type(e).__name__,
                "file_type": file_ext
            }) for e in elements if e.text.strip()]  # 过滤空文本
        except Exception as e:
            print(f"高精度解析失败，回退到Reader: {e}")
            # 回退到Reader
            reader = UnstructuredReader()
            return reader.load_data(file=file_path)

    else:
        # 简单文件或未知类型优先使用Reader
        print(f"检测到简单文件类型 {file_ext}，使用Reader解析")
        try:
            # 直接使用Reader进行简单解析
            reader = UnstructuredReader()
            # 加载解析后的文档，返回 Document 对象列表
            docs = reader.load_data(file=file_path)
            return docs
        except Exception as e:
            print(f"Reader解析失败，回退到partition: {e}")
            # 回退到底层解析
            elements = partition(filename=str(file_path), strategy="auto")
            return [Document(text=e.text, metadata={"source": str(file_path)}) for e in elements]

In [ ]:
documents = smart_load("甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf")
documents

2025-10-22 13:38:34,880 - INFO - Reading PDF for file: 甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf ...


检测到复杂文件类型 .pdf，使用partition高精度解析


[Document(id_='f4ae46d7-5370-44e9-ad9e-2a7fb1fd0835', embedding=None, metadata={'source': '甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf', 'element_type': 'Header', 'file_type': '.pdf'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='证 券 研 究 报 告 行 业 研 究', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}'),
 Document(id_='e1b21153-43ce-4bea-8f1a-2cb9bed7701a', embedding=None, metadata={'source': '甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf', 'element_type': 'Header', 'file_type': '.pdf'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='中小市值', path=None, url=None, mimetype=None), image_resour

**这种做法在实际RAG框架开发中非常常见：**
* 90%文件走LlamaIndex内置Reader
* 特殊格式（扫描件、混合HTML、表格）回退到底层`unstructured`

**总结一句话**
* **LlamaIndex的**`UnstructuredReader`** = 快速封装，适合上层应用**

* `unstructured.partition`** = 底层引擎，适合复杂数据管线**

* **在原型阶段用**`UnstructuredReader`**，在生产阶段直接集成**`unstructured`**。**


<table style="width:100%; border-collapse: collapse; margin: 20px 0;">
    <style>
        table { border: 1px solid #ddd; }
        th { background-color: #4CAF50; color: white; padding: 12px; text-align: center; border: 1px solid #ddd; font-weight: bold; }
        td { padding: 10px; text-align: center; border: 1px solid #ddd; }
        tr:nth-child(even) { background-color: #f2f2f2; }
        tr:hover { background-color: #ddd; }
    </style>
    <thead>
        <tr>
            <th>场景</th>
            <th>推荐方式</th>
            <th>理由</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>初学者/快速Demo</td>
            <td>`UnstructuredReader()`</td>
            <td>封装好，一行搞定</td>
        </tr>
        <tr>
            <td>RAG系统</td>
            <td>`UnstructuredReader()`</td>
            <td>输出直接是Document</td>
        </tr>
        <tr>
            <td>生产系统/多文件管线</td>
            <td>`partition()`</td>
            <td>可完全控制OCR/分块/过滤</td>
        </tr>
        <tr>
            <td>精细元数据追踪（页码/坐标/字体）</td>
            <td>`partition()`</td>
            <td>元数据更丰富</td>
        </tr>
    </tbody>
</table>

### 4. 基础索引案例实现

#### 基础文档解析实现

* PDF文档需要先转换成Markdown带有标签的格式，有了文本和图片等标签以后，再进行对应部分的操作，会更加的方便。

* 解析得到的`documents`可以直接用于构建LlamaIndex的向量索引，这是RAG系统的核心。

In [ ]:
#!pip install llama-index-embeddings-openai llama-index-llms-openai

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI          # 导入OpenAI LLM类
from llama_index.core.settings import Settings
from dotenv import load_dotenv
# 加载环境变量
load_dotenv()

# 设置为全局默认Embedding模型
Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
)

# 设置为全局默认 LLM
Settings.llm = OpenAI(
    model="gpt-3.5-turbo",
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
)

# 解析pdf文档
documents = smart_load("甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf")

# 构建索引
index = VectorStoreIndex.from_documents(documents)

# 生成查询引擎
query_engine = index.as_query_engine()

# 测试提问
response = query_engine.query("请用中文总结这些文档的主要内容")
print(response)

2025-10-22 14:33:53,631 - INFO - Reading PDF for file: 甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf ...


检测到复杂文件类型 .pdf，使用partition高精度解析


2025-10-22 14:34:20,074 - INFO - HTTP Request: POST https://api.fe8.cn/v1/embeddings "HTTP/1.1 200 OK"
2025-10-22 14:34:21,950 - INFO - HTTP Request: POST https://api.fe8.cn/v1/embeddings "HTTP/1.1 200 OK"
2025-10-22 14:34:24,003 - INFO - HTTP Request: POST https://api.fe8.cn/v1/chat/completions "HTTP/1.1 200 OK"


这份报告主要讨论了海外科技巨头在人工智能领域的持续投入和发展情况，重点关注了龙头公司在中报业绩方面的亮眼表现。


## 第六阶段：特定领域应用

### 金融领域：财报解析与问答

* 重点：表格提取精度
* 推荐工具：DoclingAI（表格提取精度98%+）
* 策略：使用`hi_res`策略进行高精度表格解析

### 医疗领域：文献分析与检索

* 重点：公式识别与专业术语处理
* 推荐工具：MinerU（数学公式识别精准）
* 策略：结合OCR与专业词典

### 法律领域：合同解析与条款提取

* 重点：文档结构还原与条款定位
* 策略：利用标题元素构建文档层次结构
* 元数据：页码、坐标信息用于精确定位

### 教育领域：教材内容分析与问答

* 重点：多模态内容处理（文字、图片、公式）
* 推荐工具：Marker（代码/公式支持优秀）
* 策略：使用`vlm`策略处理复杂版面

## 学习资源与参考

### 官方资源

* unstructured官网：[https://unstructured.io/](https://unstructured.io/)

* unstructured GitHub：[https://github.com/Unstructured-IO/unstructured](https://github.com/Unstructured-IO/unstructured)

* LlamaIndex官方文档：[https://docs.llamaindex.org.cn/en/stable/](https://docs.llamaindex.org.cn/en/stable/)

* Tesseract OCR官方文档：[https://tesseract-ocr.github.io/](https://tesseract-ocr.github.io/)

* pdf2image文档：[https://pdf2image.readthedocs.io/](https://pdf2image.readthedocs.io/)

### LlamaIndex 的更多功能

* 智能体（Agent）开发框架：[https://docs.llamaindex.ai/en/stable/module_guides/deploying/agents/](https://docs.llamaindex.ai/en/stable/module_guides/deploying/agents/)

* RAG 的评测：[https://docs.llamaindex.ai/en/stable/module_guides/evaluating/](https://docs.llamaindex.ai/en/stable/module_guides/evaluating/)

* 过程监控：[https://docs.llamaindex.ai/en/stable/module_guides/observability/](https://docs.llamaindex.ai/en/stable/module_guides/observability/)

此外，LlamaIndex 针对生产级的 RAG 系统中遇到的各个方面的细节问题，总结了很多高端技巧（[Advanced Topics](https://docs.llamaindex.ai/en/stable/optimizing/production_rag/)），对实战很有参考价值，非常推荐有能力的同学阅读。

**技术文档**
* AWS SDK介绍：[https://aws.amazon.com/cn/what-is/sdk/](https://aws.amazon.com/cn/what-is/sdk/)

* SDK和API的区别：[https://aws.amazon.com/cn/compare/the-difference-between-sdk-and-api/](https://aws.amazon.com/cn/compare/the-difference-between-sdk-and-api/)